In [1]:
%load_ext autoreload
%autoreload 2
from sindex.sources.openalex.snapshot import (
    run_openalex_sweep, 
    process_openalex_topics_for_datasets, 
    process_openalex_citations_for_datasets,
    export_citations_to_ndjson
)
import duckdb
import os

## Test with some files from the snapshot

In [16]:
# Path configuration
config = {
    "source_dir": r"E:\OpenAlex\snapshot-light\data\works",
    "meta_out": r"C:\Users\Admin\Documents\OpenAlex\test\metadata",
    "cite_out": r"C:\Users\Admin\Documents\OpenAlex\test\citations",
     "db_path": r"C:\Users\Admin\Documents\OpenAlex\test\openalex_test.db",
    "temp": r"C:\Users\Admin\Documents\OpenAlex\test\duckdb_temp",
}

In [4]:
# Create parquet filesfrom oa works gz files
results = run_openalex_sweep(**config, max_workers=16)

if results['errors']:
    print(f"\nCompleted with errors: {len(results['errors'])}")

Syncing 13 files...
Progress: 13/13 (New: 0, Skipped: 13)

In [5]:
# Load in DuckDB
db_path = r"C:\Users\Admin\Documents\OpenAlex\test\openalex_test.db"
con = duckdb.connect(db_path)

meta_path = f"{config['meta_out']}/*.parquet"
cite_path = f"{config['cite_out']}/*.parquet"

con.execute(f"CREATE OR REPLACE TABLE works AS SELECT * FROM read_parquet('{meta_path}')")
con.execute(f"CREATE OR REPLACE TABLE citations AS SELECT * FROM read_parquet('{cite_path}')")

meta_count = con.execute("SELECT count(*) FROM works").fetchone()[0]
cite_count = con.execute("SELECT count(*) FROM citations").fetchone()[0]

print(f"Success! Loaded {meta_count:,} works and {cite_count:,} citation links.")

Loading metadata from: C:\Users\Admin\Documents\OpenAlex\test\metadata/*.parquet
Loading citations from: C:\Users\Admin\Documents\OpenAlex\test\citations/*.parquet
Success! Loaded 408 works and 689 citation links.


In [18]:
# Preview
df_sample_work = con.execute("SELECT * FROM works ORDER BY topic_id DESC LIMIT 10").df()
display(df_sample_work)
df_sample_citations = con.execute("SELECT * FROM citations ORDER BY citing_oa_id LIMIT 20").df()
display(df_sample_citations)

,oa_id,doi,pub_date,topic_id,topic_name,topic_score
0,W1547878197,10.18372/2411-264x.3.2141,2012-09-10,T13497,Hermeneutics and Narrative Identity,0.9879
1,W2288414950,10.36418/syntax-literate.v3i3.350,2018-03-27,T13497,Hermeneutics and Narrative Identity,0.9879
2,W1843317143,10.31941/delta.v1i2.482,2017-08-30,T13497,Hermeneutics and Narrative Identity,0.9879
3,W2338400371,10.11903/1002.6495.2014.087,2015-02-14,T13497,Hermeneutics and Narrative Identity,0.9879
4,W2281025132,10.18372/2412-2157.12.8295,2015-05-23,T13497,Hermeneutics and Narrative Identity,0.9879
5,W2283374966,10.2495/str030361,2003-04-10,T13497,Hermeneutics and Narrative Identity,0.9879
6,W2328865013,,1977-01-01,T13497,Hermeneutics and Narrative Identity,0.9879
7,W2303842597,,1986-02-01,T13497,Hermeneutics and Narrative Identity,0.9879
8,W2395157951,10.33772/medula.v1i1.188,2013-01-01,T13497,Hermeneutics and Narrative Identity,0.9879
9,W2190023776,10.18372/2412-2157.16.9474,2015-11-16,T13497,Hermeneutics and Narrative Identity,0.9879


,citing_oa_id,cited_oa_id
0,W139517439,W2114449116
1,W144694350,W40440957
2,W1480302865,W1966463595
3,W1480302865,W2271065608
4,W1480302865,W2236238848
5,W1480302865,W2073521201
6,W1480302865,W2338124369
7,W1480302865,W2292183954
8,W1480302865,W2044284902
9,W1480302865,W2005121720


In [19]:
target_doi = "10.11903/1002.6495.2014.087"

query = f"""
SELECT 
    -- 1. Info about the Target Paper
    w.doi AS target_doi,
    w.topic_name AS target_topic,
    w.pub_date AS target_date,
    
    -- 2. Info about the Citing Papers
    c.citing_oa_id,
    citing_meta.doi AS citing_doi,
    citing_meta.pub_date AS citing_date,
    citing_meta.topic_name AS citing_topic
FROM works w
LEFT JOIN citations c ON w.oa_id = c.cited_oa_id
LEFT JOIN works citing_meta ON c.citing_oa_id = citing_meta.oa_id
WHERE w.doi = '{target_doi}';
"""

df_results = con.execute(query).df()
display(df_results)

,target_doi,target_topic,target_date,citing_oa_id,citing_doi,citing_date,citing_topic
0,10.11903/1002.6495.2014.087,Hermeneutics and Narrative Identity,2015-02-14,None,None,None,None


In [18]:
doi_list = ['10.18372/2411-264x.3.2141', '10.36418/syntax-literate.v3i3.350']

# Connect to DuckDB (creates a file named my_data.db)
con = duckdb.connect(config["db_path"])

# Create the table using the Python list
con.execute("CREATE TABLE target_dois AS SELECT * FROM (SELECT unnest(?) AS doi)", [doi_list])

# Verify
display(con.execute("SELECT * FROM target_dois").df())

,doi
0,10.18372/2411-264x.3.2141
1,10.36418/syntax-literate.v3i3.350


In [19]:
process_openalex_topics_for_dois(
    db_path=config["db_path"], 
    meta_folder=config["meta_out"], 
    mem_limit="32GB", 
    temp_dir=config["temp"]
)

Starting scan of 13 metadata files...
Scanned: 13/13 | Elapsed: 00:00
Finalizing: Deduplicating DOIs (keeping highest topic scores)...
Done! Unique datasets found: 2 | Total Time: 0.01 min


In [20]:
display(con.execute("SELECT * FROM my_datasets_topics").df())

,oa_id,doi,pub_date,topic_id,topic_name,topic_score
0,W1547878197,10.18372/2411-264x.3.2141,2012-09-10,T13497,Hermeneutics and Narrative Identity,0.9879
1,W2288414950,10.36418/syntax-literate.v3i3.350,2018-03-27,T13497,Hermeneutics and Narrative Identity,0.9879


In [21]:
con.close()

## Actual run full OpenAlex snapshot

### Config

In [2]:
# Path configuration
# config = {
#     "source_dir": r"E:\OpenAlex\openalex-snapshot\data\works",
#     "meta_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\metadata",
#     "cite_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\citations",
#     "db_path": r"C:\Users\Admin\Documents\OpenAlex\openalex-snapshot\openalex.db",
#     "temp": r"C:/Users/Admin/Documents/OpenAlex/openalex-snapshot/duckdb_temp",
# }
config = {
    "source_dir": r"E:\OpenAlex\openalex-snapshot\data\works",
    "meta_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\metadata",
    "cite_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\citations",
    "db_path": r"I:\pipeline-data\openalex_processed\openalex_topics_citations.db",
    "dataset_db_path": r"I:\pipeline-data\records\slim-records\datacite-slim-records.duckdb",
    "temp": r"C:/Users/Admin/Documents/OpenAlex/full-snapshot/duckdb_temp",
}

### Extract topics and citations from full snapshot in parquet files

In [21]:
# Create parquet files from oa works gz files
results = run_openalex_sweep(**config, max_workers=16)

if results['errors']:
    print(f"\nCompleted with errors: {len(results['errors'])}")

Syncing 1716 files...
Progress: 1716/1716 (New: 1716, Skipped: 0)

## Topics and citations for DOIs

### Get topics for datasets of interest

In [4]:
process_openalex_topics_for_datasets(
    db_path=config["db_path"], 
    dataset_db_path = config["dataset_db_path"],
    meta_folder = config["meta_out"], 
    mem_limit = "32GB", 
    temp_dir = config["temp"]
)

All metadata files have already been processed.

Finalizing: Computing 'best_date' and deduplicating...
Done! Unique entries: 47,476,822 | Total Time: 6.13 min


In [6]:
con =  duckdb.connect(config["db_path"])
print("\nDataset Topics")
row_count = con.execute("SELECT count() FROM my_datasets_topics").fetchone()[0]
print(f"Total DOIs in table: {row_count:,}")
display(con.execute("SELECT * FROM my_datasets_topics LIMIT 5").df())
con.close()


 Dataset Topics
Total DOIs in table: 47476822


,oa_id,doi,publication_date,created_date,topic_id,topic_name,topic_score,best_date
0,W7080713530,10.3535/9ye-v6f-np6,2025-09-09T00:00:00,2025-09-09T20:59:42+00:00,T12157,Geochemistry and Geologic Mapping,0.398812,2025-09-09T00:00:00
1,W6965796032,10.3535/9yh-aaq-dej,2025-07-04T00:00:00,2025-07-10T17:20:23+00:00,None,None,NaN,2025-07-04T00:00:00
2,W6890507870,10.3535/9yj-kvx-36p,2025-08-11T00:00:00,2025-08-12T08:07:48+00:00,None,None,NaN,2025-08-11T00:00:00
3,W6909113662,10.3535/9yk-vhv-yrl,2025-07-01T00:00:00,2025-07-01T12:43:09+00:00,T13370,Diverse Scientific and Economic Studies,0.023987,2025-07-01T00:00:00
4,W6965410726,10.3535/9ym-3hj-ex6,2025-07-04T00:00:00,2025-07-09T23:55:29+00:00,None,None,NaN,2025-07-04T00:00:00


### Get citations

In [3]:
process_openalex_citations_for_datasets(
    db_path=config["db_path"], 
    dataset_db_path = config["dataset_db_path"],
    cite_folder=config["cite_out"],
    meta_folder=config["meta_out"], 
    mem_limit="32GB", 
    temp_dir=config["temp"]
)

Step 1: Finding citations in 1716 files
 > Progress: 1716/1716 | Elapsed: 20:29

Step 2: Mapping citing IDs and calculating best_date from external DB
Done! Final citation count: 2,754,955 | Total Time: 26.52 min


In [11]:
con =  duckdb.connect(config["db_path"])
print("\n Dataset Citations")
row_count = con.execute("SELECT count() FROM my_datasets_citations").fetchone()[0]
print(f"Total citations in table: {row_count:,}")
display(con.execute("SELECT * FROM my_datasets_citations LIMIT 5").df())
con.close()


 Dataset Citations
Total citations in table: 2,754,955


,cited_doi,cited_oa_id,citing_oa_id,citing_doi,citation_date,publication_date,created_date,best_date
0,10.6084/m9.figshare.1510980.v4,W2275684564,W2514643174,10.1002/2015ea000158,2016-07-26,2016-01-01T00:00:00,2016-05-14T13:36:17+00:00,2016-01-01T00:00:00
1,10.15468/bnlkyy,W4395314871,W4396464360,10.15468/dl.hdbkml,2017-08-30,2021-01-01T00:00:00,2016-05-14T14:22:43+00:00,2021-01-01T00:00:00
2,10.6084/m9.figshare.3381109,W4394386444,W4210460784,10.1017/heq.2021.53,2022-02-01,2016-01-01T00:00:00,2016-05-14T14:33:01+00:00,2016-01-01T00:00:00
3,10.7286/v13x84k1,W6977408115,W4385759394,10.3390/biology12081124,2023-08-11,2016-01-01T00:00:00,2016-05-14T15:40:42+00:00,2016-01-01T00:00:00
4,10.6084/m9.figshare.1533109.v4,W4394558733,W4312181754,10.46966/msjar.v3i4.82,2022-12-23,2016-01-01T00:00:00,2016-05-14T19:16:37+00:00,2016-01-01T00:00:00


### Generate OpenAlex citations ndjson

In [3]:
export_citations_to_ndjson(
    db_path=config["db_path"],
    out_ndjson = r"I:\pipeline-data\citations\openalex\oa_citations.ndjson"
)

Starting export to I:\pipeline-data\citations\openalex\oa_citations.ndjson
Processed: 2,754,955 | Elapsed: 726.89s
Complete. Total citation records: 2,754,955


## Analysis

### Topics

In [20]:
db_path = r"I:\pipeline-data\openalex_processed\openalex_topics_citations.db"

In [26]:
con = duckdb.connect(db_path)
count = con.execute("""
    SELECT COUNT(*) 
    FROM my_datasets_topics 
    WHERE topic_score > 0.5
""").fetchone()[0]

print(f"Number of items with a high topic score: {count:,}")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of items with a high topic score: 6,618,256


### Citations

In [12]:
db_path = r"I:\pipeline-data\openalex_processed\openalex_topics_citations.db"

In [15]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(DISTINCT cited_doi) FROM my_datasets_citations").fetchone()[0]
print(f"Number of unique cited DOIs: {result:,}")
con.close()

Number of unique cited DOIs: 230,280


In [16]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(DISTINCT citing_oa_id) FROM my_datasets_citations").fetchone()[0]
print(f"Number of unique citing resources: {result:,}")
con.close()

Number of unique citing resources: 651,237


In [18]:
con = duckdb.connect(db_path)
query = """
    SELECT 
        cited_doi, 
        COUNT(*) AS citation_count
    FROM my_datasets_citations
    WHERE cited_doi IS NOT NULL
    GROUP BY cited_doi
    ORDER BY citation_count DESC
    LIMIT 10
"""
top_10_df = con.execute(query).df()# To see it:
display(top_10_df)
con.close()

,cited_doi,citation_count
0,10.57702/zp44cu3g,25430
1,10.15468/hnhrg3,18598
2,10.57702/kcdhx0zi,18067
3,10.5281/zenodo.5781449,16968
4,10.5281/zenodo.15145663,15746
5,10.57702/o9raffed,15636
6,10.57702/sq75seit,14599
7,10.15468/ib5ypt,11722
8,10.15468/6e8nje,10335
9,10.15468/nc6rxy,9899
